# Accord Ratings Across Fragrance Genders

## 1. Objective

This analysis explores how fragrance accord ratings differ across women's, men's, and unisex fragrances. The goal is to identify accords associated with higher ratings, understand how their performance varies across gender categories, and establish a data-driven foundation for identifying potentially underexplored fragrance opportunities.

The analysis uses the existing Fragrantica reference dataset to examine accord-level ratings alongside the number of fragrances and community engagement.

## 2. Analytical Questions

The analysis aims to answer the following questions:

- Which accords have the highest average and median ratings within each gender category?
- Which accords perform consistently well across women's, men's, and unisex fragrances?
- Which accords show notable rating differences between gender categories?
- How many fragrances are represented by each accord within each gender?
- Are highly rated accords supported by a sufficiently large sample of fragrances?
- Which accords combine strong ratings with relatively limited representation in a particular gender category?

## 3. Analytical Approach

The analysis will begin with data validation and exploration, followed by gender-specific aggregation and comparison.

For each accord and gender category, the analysis will examine:

- **Average rating:** the mean rating of fragrances associated with the accord.
- **Median rating:** the central rating, less sensitive to extreme values.
- **Fragrance count:** the number of distinct fragrances associated with the accord.
- **Rating distribution:** the spread of ratings and the consistency of observed performance.

Where supported by the available data, community engagement metrics may be incorporated as additional context.

## 4. Methodological Considerations

A fragrance can be associated with multiple accords. Consequently, accord-level fragrance counts are not additive, and the same fragrance may contribute to several accord groups.

Gender categories will follow the classifications available in the reference dataset. Any missing or inconsistent classifications will be examined before aggregation.

Results will be interpreted in the context of sample size. An accord with a high average rating but very few fragrances may warrant further investigation, but its rating is less reliable as evidence of a broader pattern.

Ratings reflect community evaluations rather than sales, market share, or demonstrated consumer purchase intent. Observed differences between gender categories are descriptive and do not establish causation.

## 5. Expected Outcome

The analysis will produce a comparative overview of accord ratings across gender categories, highlighting:

1. Highly rated accords within each category.
2. Accords with relatively consistent performance across categories.
3. Accords with notable differences between categories.
4. Highly rated accords with limited representation that may merit further investigation.

These findings will provide the basis for a subsequent opportunity analysis combining rating performance, fragrance count, and community attention.

In [0]:

%pip install --upgrade "plotly>=6.1.1" "kaleido>=1.0.0"
%pip install adjustText
from pyspark.sql import functions as F

# Source: existing cleaned Fragrantica reference dataset
SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"

fragrantica_df = spark.table(SOURCE_TABLE)

print(f"Source table: {SOURCE_TABLE}")
print(f"Total records: {fragrantica_df.count():,}")

# Inspect available columns and data types
fragrantica_df.printSchema()

# Preview records to validate the source data
display(fragrantica_df.limit(10))


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Parse accord lists and extract the first-listed accord
accords_df = (
    fragrantica_df
    .withColumn(
        "accord_array",
        F.from_json(
            F.regexp_replace(F.col("main_accords"), "'", '"'),
            "array<string>"
        )
    )
    .withColumn(
        "dominant_accord",
        F.lower(F.trim(F.get(F.col("accord_array"), 0)))
    )
    .withColumn("gender", F.lower(F.trim(F.col("gender"))))
)

# Keep fragrances with recognized gender, an accord, and a valid rating
dominant_accord_df = (
    accords_df
    .filter(F.col("gender").isin("women", "men", "unisex"))
    .filter(
        F.col("dominant_accord").isNotNull()
        & (F.col("dominant_accord") != "")
        & F.col("rating_value").isNotNull()
    )
    .dropDuplicates(["id"])
)

# Calculate rating and popularity metrics by gender and accord
dominant_accord_stats = (
    dominant_accord_df
    .groupBy("gender", "dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("mean_rating"),
        F.round(
            F.percentile_approx("rating_value", 0.5), 3
        ).alias("median_rating"),
        F.sum("rating_count").alias("total_votes"),
        F.count("rating_count").alias("fragrances_with_vote_count"),
        F.round(
            F.avg("rating_count"), 2
        ).alias("mean_votes_per_fragrance")
    )
    .withColumn(
        "rating_rank",
        F.dense_rank().over(
            Window.partitionBy("gender")
            .orderBy(F.desc("mean_rating"))
        )
    )
    .withColumn(
        "popularity_rank",
        F.dense_rank().over(
            Window.partitionBy("gender")
            .orderBy(F.desc("mean_votes_per_fragrance"))
        )
    )
)

display(
    dominant_accord_stats
    .orderBy("gender", "rating_rank")
)

In [0]:

import pandas as pd
import numpy as np

# Use the same source and cleaning rules as the bubble charts
article_df = dominant_accord_stats.toPandas()

article_df = (
    article_df
    .dropna(subset=[
        "gender",
        "dominant_accord",
        "mean_votes_per_fragrance",
        "mean_rating",
        "fragrance_count"
    ])
    .drop_duplicates(subset=["gender", "dominant_accord"])
    .copy()
)

article_df["gender"] = article_df["gender"].str.lower()
article_df["dominant_accord"] = article_df["dominant_accord"].str.title()

numeric_cols = [
    "mean_votes_per_fragrance",
    "mean_rating",
    "fragrance_count"
]

for col in numeric_cols:
    article_df[col] = pd.to_numeric(
        article_df[col], errors="coerce"
    )

article_df = article_df.dropna(subset=numeric_cols)

# Apply the exact same median-based segmentation as the charts
segment_names = {
    ("Higher popularity", "Higher rating"): "Popular Performers",
    ("Lower popularity", "Higher rating"): "High-Rated Niche Accords",
    ("Higher popularity", "Lower rating"): "Popular Underperformers",
    ("Lower popularity", "Lower rating"): "Low-Visibility, Lower-Rated Accords"
}

article_df["popularity_median"] = (
    article_df.groupby("gender")["mean_votes_per_fragrance"]
    .transform("median")
)

article_df["rating_median"] = (
    article_df.groupby("gender")["mean_rating"]
    .transform("median")
)

article_df["popularity_segment"] = np.where(
    article_df["mean_votes_per_fragrance"]
    >= article_df["popularity_median"],
    "Higher popularity",
    "Lower popularity"
)

article_df["rating_segment"] = np.where(
    article_df["mean_rating"] >= article_df["rating_median"],
    "Higher rating",
    "Lower rating"
)

article_df["segment"] = [
    segment_names[(p, r)]
    for p, r in zip(
        article_df["popularity_segment"],
        article_df["rating_segment"]
    )
]

# 1. Exact thresholds used in each gender-specific bubble chart
thresholds = (
    article_df.groupby("gender", as_index=False)
    .agg(
        accord_count=("dominant_accord", "count"),
        median_votes_per_fragrance=("popularity_median", "first"),
        median_rating=("rating_median", "first")
    )
)

print("=== GENDER-SPECIFIC MEDIAN THRESHOLDS ===")
display(thresholds.round(3))

# 2. Summary statistics for each segment
segment_order = [
    "Popular Performers",
    "High-Rated Niche Accords",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Accords"
]

summary = (
    article_df.groupby(["gender", "segment"], as_index=False)
    .agg(
        accord_count=("dominant_accord", "count"),
        median_fragrance_count=("fragrance_count", "median"),
        mean_rating=("mean_rating", "mean"),
        mean_votes_per_fragrance=("mean_votes_per_fragrance", "mean"),
        min_fragrance_count=("fragrance_count", "min"),
        max_fragrance_count=("fragrance_count", "max")
    )
)

summary["segment_order"] = summary["segment"].map(
    {name: i for i, name in enumerate(segment_order)}
)

summary = (
    summary.sort_values(["gender", "segment_order"])
    .drop(columns="segment_order")
    .reset_index(drop=True)
)

print("=== SEGMENT SUMMARY ===")
display(summary.round(3))

# 3. Every accord and its metrics, grouped by gender and segment
accord_details = article_df[
    [
        "gender",
        "segment",
        "dominant_accord",
        "fragrance_count",
        "mean_rating",
        "mean_votes_per_fragrance",
        "popularity_median",
        "rating_median"
    ]
].copy()

# Include additional source metrics when available
for col in [
    "total_votes",
    "fragrances_with_vote_count",
    "median_rating",
    "rating_rank",
    "popularity_rank"
]:
    if col in article_df.columns:
        accord_details[col] = article_df[col]

accord_details["segment_order"] = accord_details["segment"].map(
    {name: i for i, name in enumerate(segment_order)}
)

accord_details = (
    accord_details
    .sort_values([
        "gender",
        "segment_order",
        "mean_rating",
        "mean_votes_per_fragrance"
    ], ascending=[True, True, False, False])
    .drop(columns="segment_order")
    .reset_index(drop=True)
)

print("=== ACCORD-LEVEL FINDINGS ===")
display(accord_details.round(3))

In [0]:
# ============================================================
# FRAGRANCE DOMINANT ACCORDS — ARTICLE VISUALIZATIONS
# Creates and saves four separate, high-resolution PNG files
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ------------------------------------------------------------
# 1. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = "/Volumes/main/fragrance_analytics/reference_data/visualizations"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. VISUAL STYLE
# ------------------------------------------------------------

BG = "#21171f"
PANEL = "#2d202a"
TEXT = "#f4edf0"
MUTED = "#c1afb9"
GRID = "#59434f"

GENDER_COLORS = {
    "Men": "#c69b55",
    "Women": "#d49aaa",
    "Unisex": "#a98bba",
}

SEGMENT_COLORS = {
    "Popular Performers": "#c69b55",
    "High-Rated Niche Accords": "#a98bba",
    "Popular Underperformers": "#d49aaa",
    "Low-Visibility, Lower-Rated Accords": "#795b70",
}

# Keep the ordering consistent across all visualizations.
GENDER_ORDER = ["Men", "Women", "Unisex"]

segment_order = [
    "Popular Performers",
    "High-Rated Niche Accords",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Accords",
]

SEGMENT_LABELS = [
    "Popular\nPerformers",
    "High-Rated\nNiche Accords",
    "Popular\nUnderperformers",
    "Low-Visibility,\nLower-Rated Accords",
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.titlesize": 19,
    "savefig.dpi": 300,
})

# ------------------------------------------------------------
# 3. REUSABLE FORMATTING FUNCTIONS
# ------------------------------------------------------------

def style_axis(ax, title=None):
    ax.set_facecolor(PANEL)

    if title:
        ax.set_title(
            title,
            color=TEXT,
            fontsize=14,
            fontweight="medium",
            pad=14,
        )

    ax.tick_params(
        axis="both",
        colors=TEXT,
        labelsize=11,
        pad=7,
    )

    ax.grid(
        axis="y",
        color=GRID,
        alpha=0.45,
        linewidth=0.8,
    )

    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_color(GRID)

    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.xaxis.label.set_size(12)
    ax.yaxis.label.set_size(12)


def style_figure(fig, title, subtitle=None):
    fig.patch.set_facecolor(BG)

    fig.suptitle(
        title,
        color=TEXT,
        fontsize=19,
        fontweight="bold",
        x=0.5,
        y=0.98,
    )

    if subtitle:
        fig.text(
            0.5,
            0.935,
            subtitle,
            ha="center",
            va="top",
            color=MUTED,
            fontsize=11,
        )


def save_figure(fig, filename):
    """Save a figure as a high-resolution PNG."""
    path = os.path.join(OUTPUT_DIR, filename)

    fig.savefig(
        path,
        format="png",
        dpi=300,
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
        edgecolor="none",
        pad_inches=0.25,
    )

    print(f"Saved: {path}")
    plt.show()
    plt.close(fig)


# ------------------------------------------------------------
# 4. PREPARE THE PLOTTING DATA
# ------------------------------------------------------------

# Work on copies so the original analytical data remains unchanged.
threshold_plot = thresholds.copy()
summary_plot = summary.copy()

# Normalize gender labels for consistent ordering and colors.
threshold_plot["gender"] = (
    threshold_plot["gender"].astype(str).str.strip().str.title()
)

summary_plot["gender"] = (
    summary_plot["gender"].astype(str).str.strip().str.title()
)

# Enforce a stable order.
threshold_plot["gender"] = pd.Categorical(
    threshold_plot["gender"],
    categories=GENDER_ORDER,
    ordered=True,
)

summary_plot["gender"] = pd.Categorical(
    summary_plot["gender"],
    categories=GENDER_ORDER,
    ordered=True,
)

summary_plot["segment"] = pd.Categorical(
    summary_plot["segment"],
    categories=segment_order,
    ordered=True,
)

threshold_plot = threshold_plot.sort_values("gender")
summary_plot = summary_plot.sort_values(["gender", "segment"])

# ------------------------------------------------------------
# FIGURE 1 — GENDER-SPECIFIC MEDIAN THRESHOLDS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(14, 6.5),
)

style_figure(
    fig,
    "Gender-Specific Median Thresholds",
    "Each gender category is segmented using its own median values.",
)

metrics = [
    ("median_votes_per_fragrance", "Median community attention"),
    ("median_rating", "Median mean rating"),
]

for ax, (column, title) in zip(axes, metrics):
    style_axis(ax, title)

    plot_data = threshold_plot.dropna(subset=[column])

    bars = ax.bar(
        plot_data["gender"].astype(str),
        plot_data[column],
        color=[
            GENDER_COLORS.get(str(g), MUTED)
            for g in plot_data["gender"]
        ],
        width=0.62,
        edgecolor=BG,
        linewidth=0.8,
    )

    ax.set_ylabel(
        "Mean votes per fragrance" if column == "median_votes_per_fragrance"
        else "Mean rating",
        fontsize=12,
    )

    ax.set_xlabel("Gender category", fontsize=12)
    ax.grid(axis="x", visible=False)

    # Leave space above bars for value labels.
    ymax = plot_data[column].max()
    ax.set_ylim(0, ymax * 1.18 if ymax > 0 else 1)

    for bar, value in zip(bars, plot_data[column]):
        label = f"{value:.2f}" if column != "median_rating" else f"{value:.3f}"

        ax.annotate(
            label,
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            color=TEXT,
            fontsize=11,
        )

fig.subplots_adjust(
    top=0.82,
    bottom=0.16,
    left=0.08,
    right=0.97,
    wspace=0.28,
)

save_figure(fig, "01_gender_median_thresholds.png")


# ------------------------------------------------------------
# FIGURE 2 — SEGMENT COMPOSITION BY GENDER
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

style_figure(
    fig,
    "Accord Segment Composition by Gender",
    "Number of dominant accord groups assigned to each segment.",
)

style_axis(ax)

gender_segment_counts = (
    summary_plot
    .groupby(["gender", "segment"], observed=False)["accord_count"]
    .sum()
    .unstack("segment")
    .reindex(index=GENDER_ORDER, columns=segment_order)
    .fillna(0)
)

bottom = np.zeros(len(GENDER_ORDER))

for segment in segment_order:
    values = gender_segment_counts[segment].to_numpy(dtype=float)

    bars = ax.bar(
        GENDER_ORDER,
        values,
        bottom=bottom,
        label=segment,
        color=SEGMENT_COLORS[segment],
        width=0.62,
        edgecolor=BG,
        linewidth=0.8,
    )

    # Label only sufficiently large segments to prevent clutter.
    for i, (bar, value) in enumerate(zip(bars, values)):
        if value >= 5:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[i] + value / 2,
                f"{int(value)}",
                ha="center",
                va="center",
                color=BG,
                fontsize=11,
                fontweight="bold",
            )

    bottom += values

ax.set_xlabel("Gender category", fontsize=12)
ax.set_ylabel("Number of accord groups", fontsize=12)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="x", visible=False)

legend = ax.legend(
    title="Accord segment",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=False,
)

legend.get_title().set_color(TEXT)
legend.get_title().set_fontsize(12)

for label in legend.get_texts():
    label.set_color(TEXT)

fig.subplots_adjust(
    top=0.82,
    bottom=0.16,
    left=0.10,
    right=0.73,
)

save_figure(fig, "02_segment_composition.png")


# ------------------------------------------------------------
# FIGURE 3 — MEAN RATING BY SEGMENT AND GENDER
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 7.5))

style_figure(
    fig,
    "Mean Rating by Accord Segment and Gender",
    "Unweighted mean of the mean ratings across accord groups in each segment.",
)

style_axis(ax)

rating_column = "mean_rating"

x = np.arange(len(segment_order))
width = 0.24

for i, gender in enumerate(GENDER_ORDER):
    gender_data = (
        summary_plot[summary_plot["gender"] == gender]
        .set_index("segment")
        .reindex(segment_order)
    )

    values = gender_data[rating_column].to_numpy(dtype=float)
    positions = x + (i - 1) * width

    bars = ax.bar(
        positions,
        values,
        width=width,
        label=gender,
        color=GENDER_COLORS[gender],
        edgecolor=BG,
        linewidth=0.7,
    )

    for bar, value in zip(bars, values):
        if np.isfinite(value):
            ax.annotate(
                f"{value:.2f}",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                va="bottom",
                color=TEXT,
                fontsize=10,
                rotation=0,
            )

ax.set_xticks(x)
ax.set_xticklabels(
    SEGMENT_LABELS,
    rotation=0,
    ha="center",
    fontsize=11,
)

ax.set_ylabel("Mean rating", fontsize=12)
ax.set_xlabel("Accord segment", fontsize=12)
ax.set_ylim(
    0,
    max(5.0, summary_plot[rating_column].max() * 1.12),
)
ax.grid(axis="x", visible=False)

legend = ax.legend(
    title="Gender",
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=False,
)

legend.get_title().set_color(TEXT)

for label in legend.get_texts():
    label.set_color(TEXT)

fig.subplots_adjust(
    top=0.82,
    bottom=0.23,
    left=0.09,
    right=0.83,
)

save_figure(fig, "03_mean_rating_comparison.png")


# ------------------------------------------------------------
# FIGURE 4 — MEAN COMMUNITY ATTENTION BY SEGMENT AND GENDER
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 7.5))

style_figure(
    fig,
    "Community Attention by Accord Segment and Gender",
    "Mean votes per fragrance; logarithmic scale highlights differences across segments.",
)

style_axis(ax)

attention_column = "mean_votes_per_fragrance"

x = np.arange(len(segment_order))
width = 0.24

for i, gender in enumerate(GENDER_ORDER):
    gender_data = (
        summary_plot[summary_plot["gender"] == gender]
        .set_index("segment")
        .reindex(segment_order)
    )

    values = gender_data[attention_column].to_numpy(dtype=float)
    positions = x + (i - 1) * width

    bars = ax.bar(
        positions,
        values,
        width=width,
        label=gender,
        color=GENDER_COLORS[gender],
        edgecolor=BG,
        linewidth=0.7,
    )

    for bar, value in zip(bars, values):
        if np.isfinite(value) and value > 0:
            ax.annotate(
                f"{value:,.0f}",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                va="bottom",
                color=TEXT,
                fontsize=10,
                rotation=0,
            )

ax.set_xticks(x)
ax.set_xticklabels(
    SEGMENT_LABELS,
    rotation=0,
    ha="center",
    fontsize=11,
)

ax.set_ylabel("Mean votes per fragrance (log scale)", fontsize=12)
ax.set_xlabel("Accord segment", fontsize=12)

# A log scale makes smaller and larger attention values easier to compare.
ax.set_yscale("log")
ax.grid(axis="x", visible=False)

legend = ax.legend(
    title="Gender",
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=False,
)

legend.get_title().set_color(TEXT)

for label in legend.get_texts():
    label.set_color(TEXT)

fig.subplots_adjust(
    top=0.82,
    bottom=0.23,
    left=0.10,
    right=0.83,
)

save_figure(fig, "04_mean_attention_comparison.png")


# ------------------------------------------------------------
# 5. CONFIRM SAVED FILES
# ------------------------------------------------------------

print("\nAll four visualizations have been generated.")
print(f"Output directory: {OUTPUT_DIR}")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    if filename.lower().endswith(".png"):
        print(f" - {filename}")

In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from adjustText import adjust_text

# --------------------------------------------------
# FRAGRANCE INTELLIGENCE / DOMINANT ACCORD SEGMENTATION
# Unit of analysis: first-listed (dominant) accord
# Separate segmentation for each gender
# --------------------------------------------------

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/"
    "reference_data/visualizations"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

SEGMENT_ORDER = [
    "Popular Performers",
    "High-Rated Niche Accords",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Accords",
]

SEGMENT_NAMES = {
    ("Higher popularity", "Higher rating"): "Popular Performers",
    ("Lower popularity", "Higher rating"): "High-Rated Niche Accords",
    ("Higher popularity", "Lower rating"): "Popular Underperformers",
    ("Lower popularity", "Lower rating"): "Low-Visibility, Lower-Rated Accords",
}

SEGMENT_COLORS = {
    "Popular Performers": GOLD,
    "High-Rated Niche Accords": ROSE,
    "Popular Underperformers": BROWN,
    "Low-Visibility, Lower-Rated Accords": "#9b8493",
}

# Typography — matched to the duo bubble charts
TITLE_SIZE = 34
AXIS_LABEL_SIZE = 24
ACCORD_LABEL_SIZE = 17
VALUE_LABEL_SIZE = 19
TICK_LABEL_SIZE = 19
ANNOTATION_SIZE = 17

plt.rcParams["font.family"] = FONT

# --------------------------------------------------
# PREPARE DATA
# --------------------------------------------------

plot_df = dominant_accord_stats.toPandas()

plot_df = (
    plot_df
    .dropna(subset=[
        "gender",
        "dominant_accord",
        "mean_votes_per_fragrance",
        "mean_rating",
        "fragrance_count",
    ])
    .drop_duplicates(subset=["gender", "dominant_accord"])
    .copy()
)

plot_df["dominant_accord"] = (
    plot_df["dominant_accord"].str.title()
)

for col in [
    "mean_votes_per_fragrance",
    "mean_rating",
    "fragrance_count",
]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

plot_df = plot_df.dropna(
    subset=[
        "mean_votes_per_fragrance",
        "mean_rating",
        "fragrance_count",
    ]
)

# --------------------------------------------------
# CREATE ONE CHART PER GENDER
# --------------------------------------------------

all_summaries = []

for gender in ["men", "women", "unisex"]:

    df = plot_df[plot_df["gender"] == gender].copy()

    if len(df) < 4:
        print(f"Skipping {gender}: insufficient categories.")
        continue

    # --------------------------------------------------
    # GENDER-SPECIFIC MEDIANS AND SEGMENTS
    # --------------------------------------------------

    x_median = df["mean_votes_per_fragrance"].median()
    y_median = df["mean_rating"].median()

    df["popularity_segment"] = np.where(
        df["mean_votes_per_fragrance"] >= x_median,
        "Higher popularity",
        "Lower popularity",
    )

    df["rating_segment"] = np.where(
        df["mean_rating"] >= y_median,
        "Higher rating",
        "Lower rating",
    )

    df["segment"] = [
        SEGMENT_NAMES[(popularity, rating)]
        for popularity, rating in zip(
            df["popularity_segment"],
            df["rating_segment"],
        )
    ]

    # --------------------------------------------------
    # SELECT ACCORDS TO LABEL
    # Top 10 votes, bottom 10 ratings, top 10 ratings,
    # and top 5 by fragrance count
    # --------------------------------------------------

    highest_votes_df = df.nlargest(
        10, "mean_votes_per_fragrance"
    )

    lowest_ratings_df = df.nsmallest(
        10, "mean_rating"
    )

    highest_ratings_df = df.nlargest(
        10, "mean_rating"
    )

    largest_groups_df = df.nlargest(
        5, "fragrance_count"
    )

    labels_df = (
        pd.concat([
            highest_votes_df,
            lowest_ratings_df,
            highest_ratings_df,
            largest_groups_df,
        ])
        .drop_duplicates(subset=["dominant_accord"])
    )

    # --------------------------------------------------
    # BUBBLE SIZE SCALING
    # --------------------------------------------------

    counts = df["fragrance_count"].to_numpy(dtype=float)

    size_min, size_max = 65, 750

    if counts.max() == counts.min():
        bubble_sizes = np.full(
            len(df), (size_min + size_max) / 2
        )
    else:
        bubble_sizes = size_min + (
            (counts - counts.min()) /
            (counts.max() - counts.min())
        ) * (size_max - size_min)

    df["bubble_size"] = bubble_sizes

    # --------------------------------------------------
    # FIGURE
    # --------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(22, 15),
        dpi=180,
        facecolor=BG,
    )

    ax.set_facecolor(BG)

    ax.set_title(
        f"Dominant Accord Segmentation — {gender.title()}",
        color=TEXT,
        fontsize=TITLE_SIZE,
        fontweight="bold",
        pad=25,
    )

    # --------------------------------------------------
    # SCATTERPLOT
    # --------------------------------------------------

    ax.grid(
        True,
        color=GRID,
        linewidth=0.8,
        alpha=0.65,
    )
    ax.set_axisbelow(True)

    cmap = LinearSegmentedColormap.from_list(
        "plum_gold",
        [BROWN, "#80606b", GOLD],
    )

    ax.scatter(
        df["mean_votes_per_fragrance"],
        df["mean_rating"],
        s=df["bubble_size"],
        c=df["mean_rating"],
        cmap=cmap,
        alpha=0.90,
        edgecolors=BG,
        linewidths=0.9,
        zorder=3,
    )

    # --------------------------------------------------
    # MEDIAN LINES
    # --------------------------------------------------

    ax.axvline(
        x_median,
        color=MUTED,
        linestyle="--",
        linewidth=1.5,
        alpha=0.85,
        zorder=2,
    )

    ax.axhline(
        y_median,
        color=MUTED,
        linestyle="--",
        linewidth=1.5,
        alpha=0.85,
        zorder=2,
    )

    # --------------------------------------------------
    # LABELS WITH AUTOMATIC OVERLAP REDUCTION
    # --------------------------------------------------

    texts = []

    for _, row in labels_df.iterrows():

        is_highest_rated = row["dominant_accord"] in set(
            highest_ratings_df["dominant_accord"]
        )

        is_highest_votes = row["dominant_accord"] in set(
            highest_votes_df["dominant_accord"]
        )

        if is_highest_rated:
            label_color = GOLD
        elif is_highest_votes:
            label_color = ROSE
        else:
            label_color = TEXT

        text = ax.text(
            row["mean_votes_per_fragrance"],
            row["mean_rating"],
            row["dominant_accord"],
            color=label_color,
            fontsize=ACCORD_LABEL_SIZE,
            fontweight="bold",
            alpha=0.98,
            zorder=5,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor=BG,
                edgecolor=GRID,
                alpha=0.90,
                linewidth=0.6,
            ),
        )

        texts.append(text)

    if texts:
        adjust_text(
            texts,
            ax=ax,
            expand=(1.2, 1.4),
            force_text=(0.5, 0.8),
            force_static=(0.5, 0.8),
            force_pull=(0.05, 0.1),
            ensure_inside_axes=True,
            iter_lim=500,
            arrowprops=dict(
                arrowstyle="-",
                color=MUTED,
                lw=0.7,
                alpha=0.7,
            ),
        )

    # --------------------------------------------------
    # MEDIAN METRICS — UPPER RIGHT
    # --------------------------------------------------

    ax.text(
        0.98,
        0.97,
        f"Median mean votes: {x_median:,.1f}\n"
        f"Median mean rating: {y_median:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=ANNOTATION_SIZE,
        color=TEXT,
        bbox=dict(
            facecolor=BG,
            edgecolor=GRID,
            boxstyle="round,pad=0.5",
            alpha=0.95,
        ),
        zorder=6,
    )

    # --------------------------------------------------
    # AXES
    # --------------------------------------------------

    ax.set_xlabel(
        "Mean Votes per Fragrance",
        color=TEXT,
        fontsize=AXIS_LABEL_SIZE,
        labelpad=15,
    )

    ax.set_ylabel(
        "Mean Fragrance Rating",
        color=TEXT,
        fontsize=AXIS_LABEL_SIZE,
        labelpad=15,
    )

    ax.tick_params(
        axis="both",
        colors=TEXT,
        labelsize=TICK_LABEL_SIZE,
    )

    for spine in ax.spines.values():
        spine.set_color(GRID)

    # --------------------------------------------------
    # BUBBLE SIZE LEGEND — UNIFORM MARKERS
    # --------------------------------------------------

    legend_counts = sorted(set([
        int(counts.min()),
        int(np.median(counts)),
        int(counts.max()),
    ]))

    size_handles = []

    for count in legend_counts:

        if counts.max() == counts.min():
            area = (size_min + size_max) / 2
        else:
            area = size_min + (
                (count - counts.min()) /
                (counts.max() - counts.min())
            ) * (size_max - size_min)

        size_handles.append(
            Line2D(
                [0], [0],
                marker="o",
                linestyle="None",
                markerfacecolor=GOLD,
                markeredgecolor=TEXT,
                markeredgewidth=0.7,
                markersize=np.sqrt(area) / 2.5,
                label=f"{count:,}",
            )
        )

    legend = ax.legend(
        handles=size_handles,
        title="Number of Fragrances",
        loc="lower right",
        frameon=True,
        facecolor=BG,
        edgecolor=GRID,
        labelcolor=TEXT,
        title_fontsize=ANNOTATION_SIZE,
        fontsize=ANNOTATION_SIZE,
        handletextpad=0.8,
        labelspacing=0.8,
    )

    legend.get_title().set_color(TEXT)

    # --------------------------------------------------
    # LAYOUT AND SAVE
    # --------------------------------------------------

    plt.tight_layout(rect=[0, 0, 0.88, 1])

    output_path = os.path.join(
        OUTPUT_DIR,
        f"dominant-accord-segmentation-{gender}.png",
    )

    fig.savefig(
        output_path,
        dpi=200,
        facecolor=BG,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")

    # --------------------------------------------------
    # SEGMENT SUMMARY FOR THIS GENDER
    # --------------------------------------------------

    summary = (
        df.groupby("segment", as_index=False)
        .agg(
            accord_count=("dominant_accord", "count"),
            median_fragrance_count=("fragrance_count", "median"),
            mean_rating=("mean_rating", "mean"),
            mean_votes_per_fragrance=(
                "mean_votes_per_fragrance", "mean"
            ),
        )
    )

    summary.insert(0, "gender", gender)

    summary["segment_order"] = summary["segment"].map(
        {name: i for i, name in enumerate(SEGMENT_ORDER)}
    )

    summary = (
        summary.sort_values("segment_order")
        .drop(columns="segment_order")
        .reset_index(drop=True)
    )

    all_summaries.append(summary)

# --------------------------------------------------
# COMBINED SUMMARY
# --------------------------------------------------

if all_summaries:
    segment_summary = pd.concat(
        all_summaries,
        ignore_index=True,
    )
    display(segment_summary)

In [0]:

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/"
    "visualizations"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Color palette
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

# Large typography
TITLE_SIZE = 34
AXIS_LABEL_SIZE = 24
ACCORD_LABEL_SIZE = 21
VALUE_LABEL_SIZE = 19
TICK_LABEL_SIZE = 19
ANNOTATION_SIZE = 17

SEGMENT_COLORS = {
    "Popular Performers": GOLD,
    "High-Rated Niche Accords": ROSE,
    "Popular Underperformers": BROWN,
    "Low-Visibility, Lower-Rated Accords": "#9b8493",
}

SEGMENT_ORDER = [
    "Popular Performers",
    "High-Rated Niche Accords",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Accords",
]

GENDER_ORDER = ["men", "women", "unisex"]


def safe_filename(value):
    return re.sub(
        r"[^a-z0-9]+", "-", value.lower()
    ).strip("-")


# --------------------------------------------------
# 2. Prepare aggregation data
# --------------------------------------------------

accord_pdf = dominant_accord_stats.toPandas()

required_columns = [
    "gender",
    "dominant_accord",
    "mean_rating",
    "mean_votes_per_fragrance",
    "fragrance_count",
]

missing_columns = [
    col for col in required_columns
    if col not in accord_pdf.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

accord_pdf = accord_pdf.dropna(
    subset=required_columns
).copy()

accord_pdf["gender"] = (
    accord_pdf["gender"]
    .astype(str)
    .str.lower()
    .str.strip()
)

accord_pdf["dominant_accord"] = (
    accord_pdf["dominant_accord"]
    .astype(str)
    .str.strip()
)

for col in [
    "mean_rating",
    "mean_votes_per_fragrance",
    "fragrance_count",
]:
    accord_pdf[col] = pd.to_numeric(
        accord_pdf[col], errors="coerce"
    )

accord_pdf = accord_pdf.dropna(
    subset=[
        "mean_rating",
        "mean_votes_per_fragrance",
        "fragrance_count",
    ]
)

accord_pdf = accord_pdf.drop_duplicates(
    subset=["gender", "dominant_accord"]
)

# --------------------------------------------------
# 3. Rebuild gender-specific segmentation
# --------------------------------------------------

segmented_frames = []

for gender in GENDER_ORDER:
    gender_df = accord_pdf[
        accord_pdf["gender"] == gender
    ].copy()

    if gender_df.empty:
        print(f"No data found for {gender}.")
        continue

    median_votes = (
        gender_df["mean_votes_per_fragrance"].median()
    )
    median_rating = gender_df["mean_rating"].median()

    above_votes = (
        gender_df["mean_votes_per_fragrance"] >= median_votes
    )
    above_rating = (
        gender_df["mean_rating"] >= median_rating
    )

    conditions = [
        above_votes & above_rating,
        ~above_votes & above_rating,
        above_votes & ~above_rating,
        ~above_votes & ~above_rating,
    ]

    gender_df["segment"] = np.select(
        conditions,
        SEGMENT_ORDER,
        default="Unassigned",
    )

    gender_df["gender_median_votes"] = median_votes
    gender_df["gender_median_rating"] = median_rating

    segmented_frames.append(gender_df)

if not segmented_frames:
    raise ValueError(
        "No gender data found. Check dominant_accord_stats."
    )

segmented_df = pd.concat(
    segmented_frames,
    ignore_index=True,
)

# --------------------------------------------------
# 4. Generate large horizontal bar charts
# --------------------------------------------------

for gender in GENDER_ORDER:
    gender_df = segmented_df[
        segmented_df["gender"] == gender
    ].copy()

    if gender_df.empty:
        continue

    for segment in SEGMENT_ORDER:
        plot_df = gender_df[
            gender_df["segment"] == segment
        ].copy()

        if plot_df.empty:
            print(
                f"Skipping empty chart: {gender} / {segment}"
            )
            continue

        # Largest mean vote count appears at the top.
        plot_df = plot_df.sort_values(
            "mean_votes_per_fragrance",
            ascending=True,
        )

        max_votes = plot_df[
            "mean_votes_per_fragrance"
        ].max()

        label_offset = max(0.5, max_votes * 0.015)

        # Generous width and vertical spacing.
        fig_height = max(
            7,
            len(plot_df) * 0.65 + 2.5,
        )

        fig, ax = plt.subplots(
            figsize=(18, fig_height)
        )

        fig.patch.set_facecolor(BG)
        ax.set_facecolor(BG)

        bars = ax.barh(
            plot_df["dominant_accord"],
            plot_df["mean_votes_per_fragrance"],
            color=SEGMENT_COLORS[segment],
            edgecolor=BG,
            linewidth=0.7,
            height=0.72,
        )

        # Rating and fragrance count at the end of each bar.
        for bar, (_, row) in zip(
            bars, plot_df.iterrows()
        ):
            ax.text(
                bar.get_width() + label_offset,
                bar.get_y() + bar.get_height() / 2,
                (
                    f'{row["mean_rating"]:.2f} rating'
                    f'  |  {int(row["fragrance_count"]):,} fragrances'
                ),
                va="center",
                ha="left",
                color=TEXT,
                fontsize=VALUE_LABEL_SIZE,
                fontname=FONT,
            )

        # Extra room for larger bar-end labels.
        ax.set_xlim(
            0,
            max_votes + max(
                label_offset * 2,
                max_votes * 0.65,
            ),
        )

        ax.set_title(
            f"{gender.title()} — {segment}",
            color=TEXT,
            fontsize=TITLE_SIZE,
            fontname=FONT,
            pad=24,
        )

        ax.set_xlabel(
            "Mean votes per fragrance",
            color=MUTED,
            fontsize=AXIS_LABEL_SIZE,
            fontname=FONT,
            labelpad=15,
        )

        ax.set_ylabel(
            "Dominant accord",
            color=MUTED,
            fontsize=AXIS_LABEL_SIZE,
            fontname=FONT,
            labelpad=15,
        )

        ax.grid(
            axis="x",
            color=GRID,
            linestyle="--",
            linewidth=1.0,
            alpha=0.8,
        )

        ax.set_axisbelow(True)

        ax.tick_params(
            axis="x",
            colors=MUTED,
            labelsize=TICK_LABEL_SIZE,
            pad=8,
        )

        ax.tick_params(
            axis="y",
            colors=TEXT,
            labelsize=ACCORD_LABEL_SIZE,
            pad=7,
        )

        for spine in ax.spines.values():
            spine.set_color(GRID)

        ax.text(
            0.99,
            0.01,
            f"{len(plot_df)} accord categories",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            color=MUTED,
            fontsize=ANNOTATION_SIZE,
            fontname=FONT,
        )

        ax.margins(y=0.025)

        # Use a larger right margin for bar-end labels.
        plt.tight_layout(pad=2.0)

        filename = (
            f"dominant-accords-{safe_filename(gender)}-"
            f"{safe_filename(segment)}-mean-votes-bar.png"
        )

        output_path = os.path.join(
            OUTPUT_DIR, filename
        )

        plt.savefig(
            output_path,
            dpi=260,
            bbox_inches="tight",
            facecolor=BG,
        )

        plt.show()
        plt.close(fig)

        print(f"Saved: {output_path}")

print("Finished generating segment bar charts.")

In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# LOAD FRAGRANTICA DATA
# Reuse the Notebook 10 accord-parsing approach
# --------------------------------------------------

source_df = spark.table("main.fragrance_analytics.fragrantica_perfumes")

# Exclude missing and empty accord lists BEFORE parsing.
# This follows the previous notebook's approach.
rating_accords_df = (
    source_df
    .filter(
        F.col("main_accords").isNotNull()
        & (F.trim(F.col("main_accords")) != "[]")
    )
    .withColumn(
        "accords",
        F.from_json(
            F.regexp_replace(
                F.col("main_accords"),
                "'",
                '"'
            ),
            "array<string>"
        )
    )
    # Preserve the original accord order:
    # first = dominant, second = second accord, third = third accord
    .withColumn("dominant_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn("third_accord", F.get("accords", 2))
    .withColumn(
    "gender_group",
    F.when(F.lower(F.trim(F.col("gender"))) == "women", "Women")
     .when(F.lower(F.trim(F.col("gender"))) == "men", "Men")
     .when(F.lower(F.trim(F.col("gender"))) == "unisex", "Unisex")
)
    .select(
        "id",
        "gender",
        "gender_group",
        "rating_value",
        "rating_count",
        "main_accords",
        "accords",
        "dominant_accord",
        "second_accord",
        "third_accord"
    )
)

# Validate parsing before calculating duo or trio metrics
display(
    rating_accords_df.select(
        "id",
        "gender",
        "accords",
        "dominant_accord",
        "second_accord",
        "third_accord"
    ).limit(10)
)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# DUO ANALYSIS: DOMINANT + SECOND ACCORD
# --------------------------------------------------

duo_stats = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("gender_group").isin("Men", "Women", "Unisex")
        & F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy(
        "gender_group",
        "dominant_accord",
        "second_accord"
    )
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr(
            "percentile_approx(rating_count, 0.5)"
        ).alias("median_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr(
            "percentile_approx(rating_value, 0.5)"
        ).alias("median_rating"),
        F.avg(
            F.when(F.col("rating_value") >= 4.5, 1.0)
             .otherwise(0.0)
        ).alias("share_rated_4_5_plus")
    )
    .withColumn(
        "accord_pair",
        F.concat_ws(
            " + ",
            "dominant_accord",
            "second_accord"
        )
    )
    .withColumn(
        "rating_rank",
        F.rank().over(
            Window.partitionBy("gender_group")
            .orderBy(F.desc("mean_rating"))
        )
    )
    .withColumn(
        "popularity_rank",
        F.rank().over(
            Window.partitionBy("gender_group")
            .orderBy(F.desc("mean_votes"))
        )
    )
    .orderBy(
        "gender_group",
        F.desc("fragrance_count"),
        F.desc("mean_rating")
    )
)

display(duo_stats)

In [0]:
from pyspark.sql import functions as F

duo_distribution = (
    duo_stats
    .withColumn(
        "fragrance_count_group",
        F.when(F.col("fragrance_count") <= 10, "1–10 fragrances")
         .otherwise("More than 10")
    )
    .groupBy("gender_group", "fragrance_count_group")
    .agg(
        F.count("*").alias("duo_count"),
        F.sum("fragrance_count").alias("total_fragrances_across_duos")
    )
    .orderBy(
        "gender_group",
        F.when(F.col("fragrance_count_group") == "1–10 fragrances", 1).otherwise(2)
    )
)

display(duo_distribution)

In [0]:
import os
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# DUO GROUP DISTRIBUTION BY GENDER AND SIZE
# --------------------------------------------------

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

plt.rcParams["font.family"] = FONT

# Convert the existing Spark result to pandas.
dist_pd = duo_distribution.toPandas()

gender_order = ["Men", "Women", "Unisex"]
group_order = ["1–10 fragrances", "More than 10"]

dist_pd = dist_pd[
    dist_pd["gender_group"].isin(gender_order)
].copy()

# Ensure all gender/size combinations are present.
full_index = [
    (gender, group)
    for gender in gender_order
    for group in group_order
]

dist_pd = (
    dist_pd
    .set_index(["gender_group", "fragrance_count_group"])
    .reindex(full_index, fill_value=0)
    .rename_axis(["gender_group", "fragrance_count_group"])
    .reset_index()
)

dist_pd["duo_count"] = dist_pd["duo_count"].astype(int)

# Calculate percentages within each gender.
gender_totals = dist_pd.groupby("gender_group")["duo_count"].transform("sum")

dist_pd["percentage"] = np.where(
    gender_totals > 0,
    dist_pd["duo_count"] / gender_totals * 100,
    0
)

# Prepare grouped bars.
x = np.arange(len(gender_order))
bar_width = 0.34

small_counts = (
    dist_pd[dist_pd["fragrance_count_group"] == group_order[0]]
    .set_index("gender_group")
    .reindex(gender_order)["duo_count"]
    .to_numpy()
)

large_counts = (
    dist_pd[dist_pd["fragrance_count_group"] == group_order[1]]
    .set_index("gender_group")
    .reindex(gender_order)["duo_count"]
    .to_numpy()
)

small_pct = (
    dist_pd[dist_pd["fragrance_count_group"] == group_order[0]]
    .set_index("gender_group")
    .reindex(gender_order)["percentage"]
    .to_numpy()
)

large_pct = (
    dist_pd[dist_pd["fragrance_count_group"] == group_order[1]]
    .set_index("gender_group")
    .reindex(gender_order)["percentage"]
    .to_numpy()
)

fig, ax = plt.subplots(figsize=(16, 10))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

bars_small = ax.bar(
    x - bar_width / 2,
    small_counts,
    bar_width,
    label="1–10 fragrances",
    color=ROSE,
    edgecolor=TEXT,
    linewidth=0.7
)

bars_large = ax.bar(
    x + bar_width / 2,
    large_counts,
    bar_width,
    label="More than 10",
    color=GOLD,
    edgecolor=TEXT,
    linewidth=0.7
)

# Add count and percentage labels above each bar.
for bars, percentages in [
    (bars_small, small_pct),
    (bars_large, large_pct)
]:
    for bar, pct in zip(bars, percentages):
        height = bar.get_height()

        ax.annotate(
            f"{int(height):,}\n({pct:.1f}%)",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            va="bottom",
            color=TEXT,
            fontsize=16,
            fontweight="bold"
        )

ax.set_title(
    "Distribution of Accord Duo Groups by Gender and Size",
    color=TEXT,
    fontsize=28,
    fontweight="bold",
    pad=24
)

ax.set_xlabel(
    "Gender",
    color=TEXT,
    fontsize=21,
    labelpad=12
)

ax.set_ylabel(
    "Number of Duo Groups",
    color=TEXT,
    fontsize=21,
    labelpad=12
)

ax.set_xticks(x)
ax.set_xticklabels(
    gender_order,
    color=TEXT,
    fontsize=19
)

ax.tick_params(axis="y", colors=MUTED, labelsize=16)
ax.grid(axis="y", color=GRID, linewidth=0.8, alpha=0.7)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color(GRID)

legend = ax.legend(
    fontsize=16,
    facecolor=BG,
    edgecolor=GRID,
    labelcolor=TEXT
)

ax.margins(y=0.18)
plt.tight_layout()

output_path = os.path.join(
    OUTPUT_DIR,
    "accord_duo_distribution_by_gender_and_size.png"
)

plt.savefig(
    output_path,
    dpi=260,
    facecolor=BG,
    bbox_inches="tight"
)

display(fig)
plt.close(fig)

print(f"Saved: {output_path}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# DUO ANALYSIS: MORE THAN 10 FRAGRANCES
# --------------------------------------------------

duo_large_stats = duo_stats.filter(
    F.col("fragrance_count") > 10
)

gender_window = Window.partitionBy("gender_group")

duo_large_segmented = (
    duo_large_stats
    .withColumn(
        "median_mean_votes",
        F.percentile_approx("mean_votes", 0.5).over(gender_window)
    )
    .withColumn(
        "median_mean_rating",
        F.percentile_approx("mean_rating", 0.5).over(gender_window)
    )
    .withColumn(
        "performance_segment",
        F.when(
            (F.col("mean_votes") >= F.col("median_mean_votes"))
            & (F.col("mean_rating") >= F.col("median_mean_rating")),
            "Popular Performers"
        )
        .when(
            (F.col("mean_votes") < F.col("median_mean_votes"))
            & (F.col("mean_rating") >= F.col("median_mean_rating")),
            "High-Rated Niche Accords"
        )
        .when(
            (F.col("mean_votes") >= F.col("median_mean_votes"))
            & (F.col("mean_rating") < F.col("median_mean_rating")),
            "Popular Underperformers"
        )
        .otherwise("Low-Visibility, Lower-Rated Accords")
    )
    .orderBy(
        "gender_group",
        "performance_segment",
        F.desc("fragrance_count")
    )
)

display(duo_large_segmented)

In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from adjustText import adjust_text

# --------------------------------------------------
# STYLE
# --------------------------------------------------
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

SEGMENT_COLORS = {
    "Popular Performers": GOLD,
    "High-Rated Niche Accords": ROSE,
    "Popular Underperformers": BROWN,
    "Low-Visibility, Lower-Rated Accords": "#9b8493"
}

SEGMENT_ORDER = [
    "Popular Performers",
    "High-Rated Niche Accords",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Accords"
]

TITLE_SIZE = 34
AXIS_LABEL_SIZE = 24
ACCORD_LABEL_SIZE = 17
VALUE_LABEL_SIZE = 19
TICK_LABEL_SIZE = 19
ANNOTATION_SIZE = 17

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/"
    "reference_data/visualizations"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams["font.family"] = FONT

# --------------------------------------------------
# PREPARE DATA
# --------------------------------------------------
plot_df = duo_large_segmented.toPandas()

plot_df = plot_df.dropna(
    subset=[
        "gender_group",
        "dominant_accord",
        "second_accord",
        "mean_votes",
        "mean_rating",
        "fragrance_count",
        "performance_segment"
    ]
).copy()

for col in ["mean_votes", "mean_rating", "fragrance_count"]:
    plot_df[col] = pd.to_numeric(
        plot_df[col], errors="coerce"
    )

plot_df = plot_df.dropna(
    subset=["mean_votes", "mean_rating", "fragrance_count"]
)

# --------------------------------------------------
# ONE BUBBLE CHART PER GENDER
# --------------------------------------------------
for gender in ["Men", "Women", "Unisex"]:

    gender_df = plot_df[
        plot_df["gender_group"] == gender
    ].copy()

    if gender_df.empty:
        print(f"No data available for {gender}. Skipping.")
        continue

    median_votes = gender_df["mean_votes"].median()
    median_rating = gender_df["mean_rating"].median()

    fig, ax = plt.subplots(figsize=(22, 15))

    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    # --------------------------------------------------
    # BUBBLES BY PERFORMANCE SEGMENT
    # --------------------------------------------------
    for segment in SEGMENT_ORDER:

        segment_df = gender_df[
            gender_df["performance_segment"] == segment
        ]

        if segment_df.empty:
            continue

        ax.scatter(
            segment_df["mean_votes"],
            segment_df["mean_rating"],
            s=segment_df["fragrance_count"] * 18,
            c=SEGMENT_COLORS[segment],
            alpha=0.78,
            edgecolors=TEXT,
            linewidths=0.7,
            zorder=3
        )

    # --------------------------------------------------
    # MEDIAN LINES
    # --------------------------------------------------
    ax.axvline(
        median_votes,
        color=MUTED,
        linestyle="--",
        linewidth=1.5,
        alpha=0.85,
        zorder=2
    )

    ax.axhline(
        median_rating,
        color=MUTED,
        linestyle="--",
        linewidth=1.5,
        alpha=0.85,
        zorder=2
    )

   
# --------------------------------------------------
# SELECT DUOS TO LABEL
# --------------------------------------------------
    highest_votes_df = gender_df.nlargest(
        10, "mean_votes"
    )

    lowest_ratings_df = gender_df.nsmallest(
        10, "mean_rating"
    )

    highest_ratings_df = gender_df.nlargest(
        10, "mean_rating"
    )

    largest_groups_df = gender_df.nlargest(
        5, "fragrance_count"
    )

    labels_df = (
        pd.concat([
            highest_votes_df,
            lowest_ratings_df,
            highest_ratings_df,
            largest_groups_df
        ])
        .drop_duplicates(
            subset=["dominant_accord", "second_accord"]
        )
    )

    # --------------------------------------------------
    # ADD LABELS
    # --------------------------------------------------
    texts = []

    for _, row in labels_df.iterrows():

        label = (
            f"{row['dominant_accord']} + "
            f"{row['second_accord']}"
        )

        text = ax.text(
            row["mean_votes"],
            row["mean_rating"],
            label,
            color=TEXT,
            fontsize=ACCORD_LABEL_SIZE,
            fontweight="bold",
            alpha=0.98,
            zorder=5,
            bbox=dict(
                facecolor=BG,
                edgecolor="none",
                alpha=0.85,
                pad=1.5
            )
        )

        texts.append(text)

    # --------------------------------------------------
    # AUTOMATIC LABEL OVERLAP REDUCTION
    # --------------------------------------------------
    if texts:
        adjust_text(
            texts,
            ax=ax,
            expand=(1.2, 1.4),
            force_text=(0.5, 0.8),
            force_static=(0.5, 0.8),
            force_pull=(0.05, 0.1),
            ensure_inside_axes=True,
            iter_lim=500,
            arrowprops=dict(
                arrowstyle="-",
                color=MUTED,
                lw=0.7,
                alpha=0.7
            )
        )

    # --------------------------------------------------
    # TITLES AND AXES
    # --------------------------------------------------
    ax.set_title(
        f"{gender}: Dominant + Second Accord Pairs\n"
        "More Than 10 Fragrances",
        fontsize=TITLE_SIZE,
        color=TEXT,
        pad=25,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Mean Rating Votes per Fragrance",
        fontsize=AXIS_LABEL_SIZE,
        color=TEXT,
        labelpad=15
    )

    ax.set_ylabel(
        "Mean Fragrance Rating",
        fontsize=AXIS_LABEL_SIZE,
        color=TEXT,
        labelpad=15
    )

    ax.tick_params(
        axis="both",
        colors=TEXT,
        labelsize=TICK_LABEL_SIZE
    )

    ax.grid(
        True,
        color=GRID,
        linewidth=0.8,
        alpha=0.65
    )

    for spine in ax.spines.values():
        spine.set_color(GRID)

    # --------------------------------------------------
    # MEDIAN METRICS — UPPER RIGHT
    # --------------------------------------------------
    ax.text(
        0.98,
        0.97,
        f"Median mean votes: {median_votes:,.1f}\n"
        f"Median mean rating: {median_rating:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=ANNOTATION_SIZE,
        color=TEXT,
        bbox=dict(
            facecolor=BG,
            edgecolor=GRID,
            boxstyle="round,pad=0.5",
            alpha=0.95
        ),
        zorder=6
    )

    # --------------------------------------------------
    # CLEAN LEGEND — UNIFORM MARKERS
    # --------------------------------------------------
    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor=SEGMENT_COLORS[segment],
            markeredgecolor=TEXT,
            markeredgewidth=0.7,
            markersize=10,
            label=segment
        )
        for segment in SEGMENT_ORDER
        if segment in gender_df["performance_segment"].values
    ]

    legend = ax.legend(
        handles=legend_handles,
        title="Performance Segment",
        loc="lower right",
        fontsize=ANNOTATION_SIZE,
        title_fontsize=ANNOTATION_SIZE,
        frameon=True,
        facecolor=BG,
        edgecolor=GRID,
        labelcolor=TEXT,
        handletextpad=0.8,
        labelspacing=0.8
    )

    legend.get_title().set_color(TEXT)

    # --------------------------------------------------
    # LAYOUT AND SAVE
    # --------------------------------------------------
    plt.tight_layout()

    output_path = os.path.join(
        OUTPUT_DIR,
        f"accord_duos_{gender.lower()}_more_than_10_bubble.png"
    )

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
        facecolor=fig.get_facecolor()
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# 1. Filter to ordered accord pairs with >10 fragrances
# ---------------------------------------------------------

duo_large_segmented = duo_stats.filter(
    F.col("fragrance_count") > 10
)

# ---------------------------------------------------------
# 2. Calculate gender-specific medians
# ---------------------------------------------------------

gender_medians = (
    duo_large_segmented
    .groupBy("gender_group")
    .agg(
        F.percentile_approx("mean_votes", 0.5, 10000)
         .alias("median_mean_votes"),
        F.percentile_approx("mean_rating", 0.5, 10000)
         .alias("median_mean_rating")
    )
)

duo_large_segmented = duo_large_segmented.join(
    gender_medians,
    on="gender_group",
    how="left"
)

# ---------------------------------------------------------
# 3. Assign the four performance segments
# ---------------------------------------------------------

duo_large_segmented = duo_large_segmented.withColumn(
    "performance_segment",
    F.when(
        (F.col("mean_votes") >= F.col("median_mean_votes")) &
        (F.col("mean_rating") >= F.col("median_mean_rating")),
        "Popular Performers"
    ).when(
        (F.col("mean_votes") < F.col("median_mean_votes")) &
        (F.col("mean_rating") >= F.col("median_mean_rating")),
        "High-Rated Niche Accords"
    ).when(
        (F.col("mean_votes") >= F.col("median_mean_votes")) &
        (F.col("mean_rating") < F.col("median_mean_rating")),
        "Popular Underperformers"
    ).otherwise(
        "Low-Visibility, Lower-Rated Accords"
    )
)

# ---------------------------------------------------------
# 4. Summarize each segment by gender
# ---------------------------------------------------------

summary = (
    duo_large_segmented
    .groupBy("gender_group", "performance_segment")
    .agg(
        F.count("*").alias("accord_pair_count"),
        F.sum("fragrance_count").alias("fragrance_count"),
        F.round(
            F.sum(
                F.col("mean_rating") * F.col("fragrance_count")
            ) / F.sum("fragrance_count"),
            2
        ).alias("weighted_avg_rating"),
        F.round(
            F.sum(
                F.col("mean_votes") * F.col("fragrance_count")
            ) / F.sum("fragrance_count"),
            1
        ).alias("weighted_avg_votes_per_fragrance"),
        F.round(
            F.sum(
                F.col("share_rated_4_5_plus") *
                F.col("fragrance_count")
            ) / F.sum("fragrance_count"),
            3
        ).alias("weighted_share_rated_4_5_plus")
    )
)

# ---------------------------------------------------------
# 5. Order segments consistently
# ---------------------------------------------------------

segment_order = (
    F.when(F.col("performance_segment") == "Popular Performers", 1)
    .when(F.col("performance_segment") == "High-Rated Niche Accords", 2)
    .when(F.col("performance_segment") == "Popular Underperformers", 3)
    .when(
        F.col("performance_segment") ==
        "Low-Visibility, Lower-Rated Accords", 4
    )
    .otherwise(99)
)

summary = (
    summary
    .withColumn("_segment_order", segment_order)
    .orderBy("gender_group", "_segment_order")
    .drop("_segment_order")
)

display(summary)

In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOP_N = 15
MIN_FRAGRANCES = 10

GENDER_ORDER = ["Men", "Women", "Unisex"]

SEGMENTS = [
    ("Popular Performers", False),
    ("High-Rated Niche Accords", False),
    ("Popular Underperformers", True),
    ("Low-Visibility, Lower-Rated Accords", True),
]

plt.rcParams.update({
    "font.family": FONT,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
    "axes.edgecolor": GRID,
    "figure.facecolor": BG,
    "axes.facecolor": BG,
})


# ============================================================
# 2. FILTER AND ASSIGN PERFORMANCE SEGMENTS
# ============================================================

duo_filtered = duo_stats.filter(
    F.col("fragrance_count") > MIN_FRAGRANCES
)

# Remove existing threshold columns to avoid ambiguous references.
for col_name in ["median_votes", "median_rating"]:
    if col_name in duo_filtered.columns:
        duo_filtered = duo_filtered.drop(col_name)

gender_thresholds = (
    duo_filtered
    .groupBy("gender_group")
    .agg(
        F.expr("percentile_approx(mean_votes, 0.5)")
            .alias("median_votes"),
        F.expr("percentile_approx(mean_rating, 0.5)")
            .alias("median_rating"),
    )
)

duo_segmented = (
    duo_filtered
    .join(gender_thresholds, on="gender_group", how="left")
    .withColumn(
        "performance_segment",
        F.when(
            (F.col("mean_votes") >= F.col("median_votes"))
            & (F.col("mean_rating") >= F.col("median_rating")),
            "Popular Performers",
        )
        .when(
            (F.col("mean_votes") < F.col("median_votes"))
            & (F.col("mean_rating") >= F.col("median_rating")),
            "High-Rated Niche Accords",
        )
        .when(
            (F.col("mean_votes") >= F.col("median_votes"))
            & (F.col("mean_rating") < F.col("median_rating")),
            "Popular Underperformers",
        )
        .otherwise("Low-Visibility, Lower-Rated Accords"),
    )
)

plot_df = duo_segmented.toPandas()

for col in ["mean_votes", "mean_rating", "fragrance_count"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

plot_df = plot_df.dropna(
    subset=[
        "gender_group",
        "performance_segment",
        "dominant_accord",
        "second_accord",
        "mean_votes",
        "mean_rating",
        "fragrance_count",
    ]
)

plot_df["accord_pair"] = (
    plot_df["dominant_accord"].astype(str)
    + " \u2192 "
    + plot_df["second_accord"].astype(str)
)


# ============================================================
# 3. GENERATE BAR-AND-LINE CHARTS
# ============================================================

for gender in GENDER_ORDER:
    for segment, ascending in SEGMENTS:

        chart_df = plot_df[
            (plot_df["gender_group"] == gender)
            & (plot_df["performance_segment"] == segment)
        ].copy()

        if chart_df.empty:
            print(f"Skipping empty chart: {gender} | {segment}")
            continue

        # Rank by average votes, using the agreed segment-specific order.
        chart_df = (
            chart_df
            .sort_values("mean_votes", ascending=ascending)
            .head(TOP_N)
            .iloc[::-1]
            .reset_index(drop=True)
        )

        labels = chart_df["accord_pair"].tolist()
        votes = chart_df["mean_votes"].to_numpy(dtype=float)
        ratings = chart_df["mean_rating"].to_numpy(dtype=float)
        counts = chart_df["fragrance_count"].to_numpy(dtype=int)
        y = np.arange(len(chart_df))

        fig, ax1 = plt.subplots(
            figsize=(21, max(8, len(chart_df) * 0.62))
        )
        fig.patch.set_facecolor(BG)
        ax1.set_facecolor(BG)

        # ----------------------------------------------------
        # Bars: popularity = average votes per fragrance
        # ----------------------------------------------------
        bars = ax1.barh(
            y,
            votes,
            height=0.58,
            color=GOLD,
            alpha=0.90,
            edgecolor="none",
            zorder=2,
        )

        max_votes = max(float(votes.max()), 1.0)

        # Reserve room to the right for vote and sample-count labels.
        ax1.set_xlim(0, max_votes * 1.40)

        # Numeric average-vote labels at the end of each bar.
        vote_label_offset = max_votes * 0.012

        for bar, vote_value in zip(bars, votes):
            ax1.text(
                bar.get_width() + vote_label_offset,
                bar.get_y() + bar.get_height() / 2,
                f"{vote_value:,.1f}",
                va="center",
                ha="left",
                fontsize=13,
                color=GOLD,
                fontweight="bold",
                clip_on=False,
                zorder=6,
            )

        ax1.set_yticks(y)
        ax1.set_yticklabels(labels, fontsize=17, color=TEXT)

        ax1.set_xlabel(
            "Average votes per fragrance",
            fontsize=19,
            labelpad=14,
            color=GOLD,
        )

        ax1.tick_params(axis="x", labelsize=14, colors=MUTED)
        ax1.tick_params(axis="y", length=0, pad=12)

        ax1.xaxis.grid(
            True,
            color=GRID,
            linewidth=0.8,
            alpha=0.75,
        )
        ax1.set_axisbelow(True)

        # ----------------------------------------------------
        # Line: average rating on its own 0–5 scale
        # ----------------------------------------------------
        ax2 = ax1.twiny()
        ax2.set_facecolor("none")
        ax2.set_xlim(0, 5)

        ax2.plot(
            ratings,
            y,
            color=ROSE,
            linewidth=2.0,
            marker="o",
            markersize=8,
            markerfacecolor=ROSE,
            markeredgecolor=BG,
            markeredgewidth=1.2,
            zorder=5,
        )

        ax2.set_xlabel(
            "Average rating (out of 5)",
            fontsize=19,
            labelpad=14,
            color=ROSE,
        )
        ax2.set_xticks([0, 1, 2, 3, 4, 5])
        ax2.tick_params(axis="x", labelsize=14, colors=ROSE)

        # Rating labels placed beside their line markers.
        for i, rating in enumerate(ratings):
            # Keep labels inside the axis near the marker.
            if rating >= 4.65:
                offset_points = (-8, 0)
                horizontal_alignment = "right"
            else:
                offset_points = (7, 0)
                horizontal_alignment = "left"

            ax2.annotate(
                f"{rating:.2f}",
                xy=(rating, i),
                xytext=offset_points,
                textcoords="offset points",
                va="center",
                ha=horizontal_alignment,
                fontsize=13,
                fontweight="bold",
                color=ROSE,
                clip_on=True,
                zorder=7,
            )

        ax2.spines["top"].set_color(GRID)
        ax2.spines["bottom"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.spines["right"].set_visible(False)

        # ----------------------------------------------------
        # Fragrance count in a separate fixed right-side column
        # ----------------------------------------------------
        count_x = max_votes * 1.37

        for i, count in enumerate(counts):
            ax1.text(
                count_x,
                i,
                f"n={count:,}",
                va="center",
                ha="right",
                fontsize=13,
                color=MUTED,
                clip_on=False,
            )

        # ----------------------------------------------------
        # Title and legend
        # ----------------------------------------------------
        ax1.set_title(
            (
                f"{gender} | {segment}\n"
                f"Top {len(chart_df)} accord pairs · "
                f"More than {MIN_FRAGRANCES} fragrances per pair"
            ),
            fontsize=27,
            fontweight="bold",
            color=TEXT,
            loc="left",
            pad=28,
        )

        legend_items = [
            Line2D(
                [0], [0],
                color=GOLD,
                linewidth=9,
                label="Popularity: average votes per fragrance",
            ),
            Line2D(
                [0], [0],
                color=ROSE,
                linewidth=2,
                marker="o",
                markersize=7,
                label="Average rating (0–5)",
            ),
        ]

        ax1.legend(
            handles=legend_items,
            loc="lower right",
            bbox_to_anchor=(1.0, -0.20),
            frameon=False,
            fontsize=14,
            labelcolor=TEXT,
            ncol=2,
            borderaxespad=0,
        )

        for spine in ["top", "right", "left"]:
            ax1.spines[spine].set_visible(False)

        ax1.spines["bottom"].set_color(GRID)
        ax1.invert_yaxis()

        plt.subplots_adjust(
            left=0.28,
            right=0.95,
            top=0.84,
            bottom=0.20,
        )

        # ----------------------------------------------------
        # Save chart
        # ----------------------------------------------------
        gender_file = gender.lower().replace(" ", "_")
        segment_file = (
            segment.lower()
            .replace(" ", "_")
            .replace(",", "")
            .replace("-", "_")
        )

        filename = (
            f"accord_duos_{gender_file}_{segment_file}_"
            "top15_by_votes_rating.png"
        )

        filepath = os.path.join(OUTPUT_DIR, filename)

        fig.savefig(
            filepath,
            dpi=220,
            facecolor=BG,
            bbox_inches="tight",
        )

        plt.show()
        plt.close(fig)

        print(f"Saved: {filepath}")

print("All available accord-pair charts generated.")

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# TRIAD ANALYSIS: DOMINANT + SECOND + THIRD ACCORD
# --------------------------------------------------

triad_stats = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("gender_group").isin("Men", "Women", "Unisex")
        & F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy(
        "gender_group",
        "dominant_accord",
        "second_accord",
        "third_accord"
    )
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr(
            "percentile_approx(rating_count, 0.5)"
        ).alias("median_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr(
            "percentile_approx(rating_value, 0.5)"
        ).alias("median_rating"),
        F.avg(
            F.when(F.col("rating_value") >= 4.5, 1.0)
             .otherwise(0.0)
        ).alias("share_rated_4_5_plus")
    )
    .withColumn(
        "accord_triad",
        F.concat_ws(
            " + ",
            "dominant_accord",
            "second_accord",
            "third_accord"
        )
    )
    .withColumn(
        "rating_rank",
        F.rank().over(
            Window.partitionBy("gender_group")
            .orderBy(F.desc("mean_rating"))
        )
    )
    .withColumn(
        "popularity_rank",
        F.rank().over(
            Window.partitionBy("gender_group")
            .orderBy(F.desc("mean_votes"))
        )
    )
    .orderBy(
        "gender_group",
        F.desc("fragrance_count"),
        F.desc("mean_rating")
    )
)

display(triad_stats)

In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# TRIAD ANALYSIS: FREQUENCY AND SAMPLE-SIZE CHECK
# --------------------------------------------------

triad_frequency_summary = (
    triad_stats
    .groupBy("gender_group")
    .agg(
        F.count("*").alias("distinct_triads"),
        F.expr(
            "percentile_approx(fragrance_count, 0.25)"
        ).alias("q1_fragrance_count"),
        F.expr(
            "percentile_approx(fragrance_count, 0.50)"
        ).alias("median_fragrance_count"),
        F.expr(
            "percentile_approx(fragrance_count, 0.75)"
        ).alias("q3_fragrance_count"),
        F.max("fragrance_count").alias("max_fragrance_count"),
        F.sum(
            F.when(F.col("fragrance_count") >= 10, 1)
             .otherwise(0)
        ).alias("triads_with_10_plus"),
        F.sum(
            F.when(F.col("fragrance_count") >= 20, 1)
             .otherwise(0)
        ).alias("triads_with_20_plus"),
        F.sum(
            F.when(F.col("fragrance_count") >= 30, 1)
             .otherwise(0)
        ).alias("triads_with_30_plus"),
        F.sum(
            F.when(F.col("fragrance_count") >= 50, 1)
             .otherwise(0)
        ).alias("triads_with_50_plus")
    )
    .orderBy("gender_group")
)

display(triad_frequency_summary)

In [0]:
# TRIAD ANALYSIS: FILTER TO TRIADS WITH AT LEAST 10 FRAGRANCES

MIN_FRAGRANCES = 10

triad_stats_filtered = (
    triad_stats
    .filter(F.col("fragrance_count") >= MIN_FRAGRANCES)
)

display(
    triad_stats_filtered
    .groupBy("gender_group")
    .agg(
        F.count("*").alias("triad_count"),
        F.sum("fragrance_count").alias("fragrance_count_across_triads")
    )
    .orderBy("gender_group")
)

In [0]:
# TRIAD ANALYSIS: CHECK METRICS BEFORE VISUALIZATION

display(
    triad_stats_filtered
    .groupBy("gender_group")
    .agg(
        F.min("fragrance_count").alias("min_fragrance_count"),
        F.max("fragrance_count").alias("max_fragrance_count"),
        F.min("mean_rating").alias("min_mean_rating"),
        F.max("mean_rating").alias("max_mean_rating"),
        F.min("mean_votes").alias("min_mean_votes"),
        F.max("mean_votes").alias("max_mean_votes"),
        F.min("share_rated_4_5_plus").alias("min_share_rated_4_5_plus"),
        F.max("share_rated_4_5_plus").alias("max_share_rated_4_5_plus")
    )
    .orderBy("gender_group")
)

In [0]:

from pyspark.sql import functions as F

triad_frequency_distribution = (
    triad_stats
    .withColumn(
        "frequency_range",
        F.when(F.col("fragrance_count") == 1, "1 fragrance")
         .when(F.col("fragrance_count").between(2, 4), "2–4 fragrances")
         .when(F.col("fragrance_count").between(5, 9), "5–9 fragrances")
         .when(F.col("fragrance_count").between(10, 19), "10–19 fragrances")
         .when(F.col("fragrance_count").between(20, 49), "20–49 fragrances")
         .otherwise("50+ fragrances")
    )
    .groupBy("gender_group", "frequency_range")
    .agg(F.count("*").alias("triad_count"))
)

display(
    triad_frequency_distribution
    .orderBy("gender_group", "frequency_range")
)


In [0]:

# TRIAD ANALYSIS: PREPARE DATA FOR ALL GENDERS

triad_chart_data = (
    triad_stats_filtered
    .select(
        "gender_group",
        "dominant_accord",
        "second_accord",
        "third_accord",
        "accord_triad",
        "fragrance_count",
        "mean_rating",
        "mean_votes",
        "share_rated_4_5_plus"
    )
    .orderBy(
        "gender_group",
        F.desc("fragrance_count")
    )
)

display(triad_chart_data)

In [0]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from adjustText import adjust_text

# --------------------------------------------------
# STYLE — MATCH THE DYAD VISUALIZATIONS
# --------------------------------------------------
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

SEGMENT_COLORS = {
    "Popular Performers": GOLD,
    "High-Rated Niche Accords": ROSE,
    "Popular Underperformers": BROWN,
    "Low-Visibility, Lower-Rated Accords": "#9b8493"
}

SEGMENT_ORDER = [
    "Popular Performers",
    "High-Rated Niche Accords",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Accords"
]

TITLE_SIZE = 34
AXIS_LABEL_SIZE = 24
ACCORD_LABEL_SIZE = 17
TICK_LABEL_SIZE = 19
ANNOTATION_SIZE = 17

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/"
    "reference_data/visualizations"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams["font.family"] = FONT

# --------------------------------------------------
# PREPARE DATA
# --------------------------------------------------
plot_df = triad_stats_filtered.toPandas()

plot_df = plot_df.dropna(
    subset=[
        "gender_group",
        "dominant_accord",
        "second_accord",
        "third_accord",
        "mean_votes",
        "mean_rating",
        "fragrance_count"
    ]
).copy()

for col in ["mean_votes", "mean_rating", "fragrance_count"]:
    plot_df[col] = pd.to_numeric(
        plot_df[col], errors="coerce"
    )

plot_df = plot_df.dropna(
    subset=["mean_votes", "mean_rating", "fragrance_count"]
)

# --------------------------------------------------
# ONE BUBBLE CHART PER GENDER
# --------------------------------------------------
for gender in ["Men", "Women", "Unisex"]:

    gender_df = plot_df[
        plot_df["gender_group"] == gender
    ].copy()

    if gender_df.empty:
        print(f"No data available for {gender}. Skipping.")
        continue

    # --------------------------------------------------
    # PERFORMANCE SEGMENTS — CALCULATED PER GENDER
    # --------------------------------------------------
    median_votes = gender_df["mean_votes"].median()
    median_rating = gender_df["mean_rating"].median()

    popular = gender_df["mean_votes"] >= median_votes
    highly_rated = gender_df["mean_rating"] >= median_rating

    gender_df["performance_segment"] = np.select(
        [
            popular & highly_rated,
            ~popular & highly_rated,
            popular & ~highly_rated,
            ~popular & ~highly_rated
        ],
        SEGMENT_ORDER,
        default=SEGMENT_ORDER[3]
    )

    fig, ax = plt.subplots(figsize=(22, 15))

    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    # --------------------------------------------------
    # BUBBLES BY PERFORMANCE SEGMENT
    # --------------------------------------------------
    for segment in SEGMENT_ORDER:

        segment_df = gender_df[
            gender_df["performance_segment"] == segment
        ]

        if segment_df.empty:
            continue

        ax.scatter(
            segment_df["mean_votes"],
            segment_df["mean_rating"],
            s=segment_df["fragrance_count"] * 18,
            c=SEGMENT_COLORS[segment],
            alpha=0.78,
            edgecolors=TEXT,
            linewidths=0.7,
            zorder=3
        )

    # --------------------------------------------------
    # MEDIAN LINES
    # --------------------------------------------------
    ax.axvline(
        median_votes,
        color=MUTED,
        linestyle="--",
        linewidth=1.5,
        alpha=0.85,
        zorder=2
    )

    ax.axhline(
        median_rating,
        color=MUTED,
        linestyle="--",
        linewidth=1.5,
        alpha=0.85,
        zorder=2
    )

    # --------------------------------------------------
    # SELECT TRIADS TO LABEL
    # --------------------------------------------------
    highest_votes_df = gender_df.nlargest(
        10, "mean_votes"
    )

    lowest_ratings_df = gender_df.nsmallest(
        10, "mean_rating"
    )

    highest_ratings_df = gender_df.nlargest(
        10, "mean_rating"
    )

    largest_groups_df = gender_df.nlargest(
        5, "fragrance_count"
    )

    labels_df = (
        pd.concat([
            highest_votes_df,
            lowest_ratings_df,
            highest_ratings_df,
            largest_groups_df
        ])
        .drop_duplicates(
            subset=[
                "dominant_accord",
                "second_accord",
                "third_accord"
            ]
        )
    )

    # --------------------------------------------------
    # ADD LABELS
    # --------------------------------------------------
    texts = []

    for _, row in labels_df.iterrows():

        label = (
            f"{row['dominant_accord']} + "
            f"{row['second_accord']} + "
            f"{row['third_accord']}"
        )

        text = ax.text(
            row["mean_votes"],
            row["mean_rating"],
            label,
            color=TEXT,
            fontsize=ACCORD_LABEL_SIZE,
            fontweight="bold",
            alpha=0.98,
            zorder=5,
            bbox=dict(
                facecolor=BG,
                edgecolor="none",
                alpha=0.85,
                pad=1.5
            )
        )

        texts.append(text)

    # --------------------------------------------------
    # AUTOMATIC LABEL OVERLAP REDUCTION
    # --------------------------------------------------
    if texts:
        adjust_text(
            texts,
            ax=ax,
            expand=(1.2, 1.4),
            force_text=(0.5, 0.8),
            force_static=(0.5, 0.8),
            force_pull=(0.05, 0.1),
            ensure_inside_axes=True,
            iter_lim=500,
            arrowprops=dict(
                arrowstyle="-",
                color=MUTED,
                lw=0.7,
                alpha=0.7
            )
        )

    # --------------------------------------------------
    # TITLES AND AXES
    # --------------------------------------------------
    ax.set_title(
        f"{gender}: Dominant + Second + Third Accord Triads\n"
        "At Least 10 Fragrances",
        fontsize=TITLE_SIZE,
        color=TEXT,
        pad=25,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Mean Votes per Fragrance",
        fontsize=AXIS_LABEL_SIZE,
        color=TEXT,
        labelpad=15
    )

    ax.set_ylabel(
        "Mean Rating per Fragrance",
        fontsize=AXIS_LABEL_SIZE,
        color=TEXT,
        labelpad=15
    )

    ax.tick_params(
        axis="both",
        colors=TEXT,
        labelsize=TICK_LABEL_SIZE
    )

    ax.grid(
        True,
        color=GRID,
        linewidth=0.8,
        alpha=0.65
    )

    for spine in ax.spines.values():
        spine.set_color(GRID)

    # --------------------------------------------------
    # MEDIAN METRICS — UPPER RIGHT
    # --------------------------------------------------
    ax.text(
        0.98,
        0.97,
        f"Median mean votes: {median_votes:,.1f}\n"
        f"Median mean rating: {median_rating:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=ANNOTATION_SIZE,
        color=TEXT,
        bbox=dict(
            facecolor=BG,
            edgecolor=GRID,
            boxstyle="round,pad=0.5",
            alpha=0.95
        ),
        zorder=6
    )

    # --------------------------------------------------
    # LEGEND — UNIFORM MARKERS
    # --------------------------------------------------
    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor=SEGMENT_COLORS[segment],
            markeredgecolor=TEXT,
            markeredgewidth=0.7,
            markersize=10,
            label=segment
        )
        for segment in SEGMENT_ORDER
        if segment in gender_df["performance_segment"].values
    ]

    legend = ax.legend(
        handles=legend_handles,
        title="Performance Segment",
        loc="lower right",
        fontsize=ANNOTATION_SIZE,
        title_fontsize=ANNOTATION_SIZE,
        frameon=True,
        facecolor=BG,
        edgecolor=GRID,
        labelcolor=TEXT,
        handletextpad=0.8,
        labelspacing=0.8
    )

    legend.get_title().set_color(TEXT)

    # --------------------------------------------------
    # LAYOUT AND SAVE
    # --------------------------------------------------
    plt.tight_layout()

    output_path = os.path.join(
        OUTPUT_DIR,
        f"accord_triads_{gender.lower()}_at_least_10_bubble.png"
    )

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
        facecolor=fig.get_facecolor()
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")

In [0]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOP_N = 15
MIN_FRAGRANCES = 10

GENDER_ORDER = ["Men", "Women", "Unisex"]

SEGMENTS = [
    ("Popular Performers", False),
    ("High-Rated Niche Accords", False),
    ("Popular Underperformers", True),
    ("Low-Visibility, Lower-Rated Accords", True),
]

plt.rcParams.update({
    "font.family": FONT,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
    "axes.edgecolor": GRID,
    "figure.facecolor": BG,
    "axes.facecolor": BG,
})


# ============================================================
# 2. FILTER AND ASSIGN PERFORMANCE SEGMENTS
# ============================================================

triad_filtered = triad_stats_filtered.filter(
    F.col("fragrance_count") > MIN_FRAGRANCES
)

# Remove existing threshold columns to avoid ambiguous references.
for col_name in ["median_votes", "median_rating"]:
    if col_name in triad_filtered.columns:
        triad_filtered = triad_filtered.drop(col_name)

# Calculate separate median thresholds for each gender.
gender_thresholds = (
    triad_filtered
    .groupBy("gender_group")
    .agg(
        F.expr("percentile_approx(mean_votes, 0.5)")
        .alias("median_votes"),
        F.expr("percentile_approx(mean_rating, 0.5)")
        .alias("median_rating"),
    )
)

triad_segmented = (
    triad_filtered
    .join(gender_thresholds, on="gender_group", how="left")
    .withColumn(
        "performance_segment",
        F.when(
            (F.col("mean_votes") >= F.col("median_votes"))
            & (F.col("mean_rating") >= F.col("median_rating")),
            "Popular Performers",
        )
        .when(
            (F.col("mean_votes") < F.col("median_votes"))
            & (F.col("mean_rating") >= F.col("median_rating")),
            "High-Rated Niche Accords",
        )
        .when(
            (F.col("mean_votes") >= F.col("median_votes"))
            & (F.col("mean_rating") < F.col("median_rating")),
            "Popular Underperformers",
        )
        .otherwise("Low-Visibility, Lower-Rated Accords"),
    )
)

plot_df = triad_segmented.toPandas()

for col in ["mean_votes", "mean_rating", "fragrance_count"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

plot_df = plot_df.dropna(
    subset=[
        "gender_group",
        "performance_segment",
        "dominant_accord",
        "second_accord",
        "third_accord",
        "mean_votes",
        "mean_rating",
        "fragrance_count",
    ]
)

# Create readable labels for the three-accord combinations.
plot_df["accord_triad_label"] = (
    plot_df["dominant_accord"].astype(str)
    + " → "
    + plot_df["second_accord"].astype(str)
    + " → "
    + plot_df["third_accord"].astype(str)
)


# ============================================================
# 3. GENERATE BAR-AND-LINE CHARTS
# ============================================================

for gender in GENDER_ORDER:
    for segment, ascending in SEGMENTS:

        chart_df = plot_df[
            (plot_df["gender_group"] == gender)
            & (plot_df["performance_segment"] == segment)
        ].copy()

        if chart_df.empty:
            print(f"Skipping empty chart: {gender} | {segment}")
            continue

        # Rank by average votes using the segment-specific order.
        chart_df = (
            chart_df
            .sort_values("mean_votes", ascending=ascending)
            .head(TOP_N)
            .sort_values("mean_votes", ascending=False)
            .reset_index(drop=True)
        )

        labels = chart_df["accord_triad_label"].tolist()
        votes = chart_df["mean_votes"].to_numpy(dtype=float)
        ratings = chart_df["mean_rating"].to_numpy(dtype=float)
        counts = chart_df["fragrance_count"].to_numpy(dtype=int)
        y = np.arange(len(chart_df))

        fig, ax1 = plt.subplots(
            figsize=(21, max(8, len(chart_df) * 0.62))
        )
        fig.patch.set_facecolor(BG)
        ax1.set_facecolor(BG)

        # ----------------------------------------------------
        # Bars: popularity = average votes per fragrance
        # ----------------------------------------------------
        bars = ax1.barh(
            y,
            votes,
            height=0.58,
            color=GOLD,
            alpha=0.90,
            edgecolor="none",
            zorder=2,
        )

        max_votes = max(float(votes.max()), 1.0)

        # Reserve space for vote and sample-count labels.
        ax1.set_xlim(0, max_votes * 1.40)
        vote_label_offset = max_votes * 0.012

        for bar, vote_value in zip(bars, votes):
            ax1.text(
                bar.get_width() + vote_label_offset,
                bar.get_y() + bar.get_height() / 2,
                f"{vote_value:,.1f}",
                va="center",
                ha="left",
                fontsize=13,
                color=GOLD,
                fontweight="bold",
                clip_on=False,
                zorder=6,
            )

        ax1.set_yticks(y)
        ax1.set_yticklabels(labels, fontsize=17, color=TEXT)

        ax1.set_xlabel(
            "Average votes per fragrance",
            fontsize=19,
            labelpad=14,
            color=GOLD,
        )

        ax1.tick_params(axis="x", labelsize=14, colors=MUTED)
        ax1.tick_params(axis="y", length=0, pad=12)

        ax1.xaxis.grid(
            True,
            color=GRID,
            linewidth=0.8,
            alpha=0.75,
        )
        ax1.set_axisbelow(True)

        # ----------------------------------------------------
        # Line: average rating on its own 0–5 scale
        # ----------------------------------------------------
        ax2 = ax1.twiny()
        ax2.set_facecolor("none")
        ax2.set_xlim(0, 5)

        ax2.plot(
            ratings,
            y,
            color=ROSE,
            linewidth=2.0,
            marker="o",
            markersize=8,
            markerfacecolor=ROSE,
            markeredgecolor=BG,
            markeredgewidth=1.2,
            zorder=5,
        )

        ax2.set_xlabel(
            "Average rating (out of 5)",
            fontsize=19,
            labelpad=14,
            color=ROSE,
        )
        ax2.set_xticks([0, 1, 2, 3, 4, 5])
        ax2.tick_params(axis="x", labelsize=14, colors=ROSE)

        # Rating labels beside the markers.
        for i, rating in enumerate(ratings):
            if rating >= 4.65:
                offset_points = (-8, 0)
                horizontal_alignment = "right"
            else:
                offset_points = (7, 0)
                horizontal_alignment = "left"

            ax2.annotate(
                f"{rating:.2f}",
                xy=(rating, i),
                xytext=offset_points,
                textcoords="offset points",
                va="center",
                ha=horizontal_alignment,
                fontsize=13,
                fontweight="bold",
                color=ROSE,
                clip_on=True,
                zorder=7,
            )

        ax2.spines["top"].set_color(GRID)
        ax2.spines["bottom"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.spines["right"].set_visible(False)

        # ----------------------------------------------------
        # Fragrance count in a fixed right-side column
        # ----------------------------------------------------
        count_x = max_votes * 1.37

        for i, count in enumerate(counts):
            ax1.text(
                count_x,
                i,
                f"n={count:,}",
                va="center",
                ha="right",
                fontsize=13,
                color=MUTED,
                clip_on=False,
            )

        # ----------------------------------------------------
        # Title and legend
        # ----------------------------------------------------
        ax1.set_title(
            (
                f"{gender} | {segment}\n"
                f"Top {len(chart_df)} accord triads · "
                f"More than {MIN_FRAGRANCES} fragrances per triad"
            ),
            fontsize=27,
            fontweight="bold",
            color=TEXT,
            loc="left",
            pad=28,
        )

        legend_items = [
            Line2D(
                [0], [0],
                color=GOLD,
                linewidth=9,
                label="Popularity: average votes per fragrance",
            ),
            Line2D(
                [0], [0],
                color=ROSE,
                linewidth=2,
                marker="o",
                markersize=7,
                label="Average rating (0–5)",
            ),
        ]

        ax1.legend(
            handles=legend_items,
            loc="lower right",
            bbox_to_anchor=(1.0, -0.20),
            frameon=False,
            fontsize=14,
            labelcolor=TEXT,
            ncol=2,
            borderaxespad=0,
        )

        for spine in ["top", "right", "left"]:
            ax1.spines[spine].set_visible(False)

        ax1.spines["bottom"].set_color(GRID)
        ax1.invert_yaxis()

        plt.subplots_adjust(
            left=0.34,
            right=0.95,
            top=0.84,
            bottom=0.20,
        )

        # ----------------------------------------------------
        # Save chart
        # ----------------------------------------------------
        gender_file = gender.lower().replace(" ", "_")
        segment_file = (
            segment.lower()
            .replace(" ", "_")
            .replace(",", "")
            .replace("-", "_")
        )

        filename = (
            f"accord_triads_{gender_file}_{segment_file}_"
            "top15_by_votes_rating.png"
        )

        filepath = os.path.join(OUTPUT_DIR, filename)

        fig.savefig(
            filepath,
            dpi=220,
            facecolor=BG,
            bbox_inches="tight",
        )

        plt.show()
        plt.close(fig)

        print(f"Saved: {filepath}")

print("All available accord-triad charts generated.")

In [0]:

import pandas as pd
from pyspark.sql import functions as F

# Match the existing top-15 charts:
# The charts use triads with MORE than 10 fragrances.
MIN_FRAGRANCES = 10

metrics = [
    "gender_group",
    "dominant_accord",
    "second_accord",
    "third_accord",
    "accord_triad",
    "fragrance_count",
    "mean_rating",
    "mean_votes",
    "share_rated_4_5_plus"
]

pdf = (
    triad_stats_filtered
    .filter(F.col("fragrance_count") > MIN_FRAGRANCES)
    .select(*metrics)
    .toPandas()
)

# Ensure metric columns are numeric.
numeric_cols = [
    "fragrance_count",
    "mean_rating",
    "mean_votes",
    "share_rated_4_5_plus"
]
for col in numeric_cols:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

segment_order = [
    "Popular Performers",
    "High-Rated Niche Accords",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Accords"
]

# Recalculate gender-specific medians on the same population
# used for selecting the chart's top 15.
pdf["performance_segment"] = None

for gender in ["Men", "Women", "Unisex"]:
    mask = pdf["gender_group"] == gender
    gender_df = pdf.loc[mask]

    votes_median = gender_df["mean_votes"].median()
    rating_median = gender_df["mean_rating"].median()

    high_votes = gender_df["mean_votes"] >= votes_median
    high_rating = gender_df["mean_rating"] >= rating_median

    labels = pd.Series(index=gender_df.index, dtype="object")
    labels.loc[high_votes & high_rating] = "Popular Performers"
    labels.loc[~high_votes & high_rating] = "High-Rated Niche Accords"
    labels.loc[high_votes & ~high_rating] = "Popular Underperformers"
    labels.loc[~high_votes & ~high_rating] = (
        "Low-Visibility, Lower-Rated Accords"
    )

    pdf.loc[mask, "performance_segment"] = labels

# Select the same 15 triads used in each chart:
# - highest mean votes for the first two segments
# - lowest mean votes for the last two segments
# Display all selected triads from highest to lowest mean votes.
top15_parts = []

for gender in ["Men", "Women", "Unisex"]:
    for segment in segment_order:
        subset = pdf[
            (pdf["gender_group"] == gender)
            & (pdf["performance_segment"] == segment)
        ].copy()

        if subset.empty:
            continue

        if segment in [
            "Popular Performers",
            "High-Rated Niche Accords"
        ]:
            selected = subset.nlargest(15, "mean_votes")
            selection_basis = "Highest mean votes"
        else:
            selected = subset.nsmallest(15, "mean_votes")
            selection_basis = "Lowest mean votes"

        selected = selected.sort_values(
            "mean_votes", ascending=False
        ).copy()

        selected["chart_order"] = range(1, len(selected) + 1)
        selected["selection_basis"] = selection_basis
        top15_parts.append(selected)

top15_results = pd.concat(top15_parts, ignore_index=True)

# Round values for readable output.
top15_results["mean_rating"] = top15_results["mean_rating"].round(3)
top15_results["mean_votes"] = top15_results["mean_votes"].round(1)
top15_results["share_rated_4_5_plus"] = (
    top15_results["share_rated_4_5_plus"].round(3)
)

# OUTPUT 1: All selected triads for the 12 charts.
display(
    top15_results[
        [
            "gender_group",
            "performance_segment",
            "chart_order",
            "accord_triad",
            "fragrance_count",
            "mean_rating",
            "mean_votes",
            "share_rated_4_5_plus"
        ]
    ].sort_values(
        ["gender_group", "performance_segment", "chart_order"]
    )
)

# OUTPUT 2: Compare the selected top-15 sets across genders
# within each performance segment.
segment_comparison = (
    top15_results
    .groupby(["performance_segment", "gender_group"], as_index=False)
    .agg(
        selected_triads=("accord_triad", "count"),
        avg_rating=("mean_rating", "mean"),
        median_rating=("mean_rating", "median"),
        avg_votes=("mean_votes", "mean"),
        median_votes=("mean_votes", "median"),
        avg_fragrance_count=("fragrance_count", "mean"),
        avg_share_rated_4_5_plus=("share_rated_4_5_plus", "mean")
    )
)

for col in [
    "avg_rating", "median_rating",
    "avg_votes", "median_votes",
    "avg_fragrance_count", "avg_share_rated_4_5_plus"
]:
    segment_comparison[col] = segment_comparison[col].round(3)

display(
    segment_comparison.sort_values(
        ["performance_segment", "gender_group"]
    )
)

# OUTPUT 3: Exact accord triads appearing in the selected top-15
# sets for two or more gender groups.
shared_triads = (
    top15_results
    .groupby("accord_triad", as_index=False)
    .agg(
        gender_groups=("gender_group", lambda s: ", ".join(sorted(set(s)))),
        gender_count=("gender_group", "nunique"),
        segments=("performance_segment", lambda s: " | ".join(sorted(set(s)))),
        fragrance_counts=("fragrance_count", lambda s: ", ".join(map(str, s))),
        ratings=("mean_rating", lambda s: ", ".join(f"{v:.3f}" for v in s)),
        average_votes=("mean_votes", lambda s: ", ".join(f"{v:.1f}" for v in s))
    )
    .query("gender_count >= 2")
    .sort_values(["gender_count", "accord_triad"], ascending=[False, True])
)

display(shared_triads)